[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/mechanics/earthquake/earthquake.ipynb)

# Earthquake Beachballs

When a fault slips, the ground on one side lurches past the other, and seismometers all around the globe feel the first wave arrive either as a push or as a pull. Which one they feel depends on the direction they lie in, and seismologists summarize the pattern in a beachball: the lower half of the sphere of directions, flattened, with the pushes shaded. This notebook writes the slip as a moment, a map on directions built from the fault's normal and its slip, reads the beachball off it, and recovers the fault from the moment's eigenvectors, up to one ambiguity that no seismometer can resolve.

In the tensor notation of seismology the moment reads as $M_{ij} = M_0\,(n_i s_j + n_j s_i)$, and the first motion of the P wave along the unit direction $\gamma$ as $\gamma_i M_{ij} \gamma_j$.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np
from IPython.display import Image, display

from numga import NumpyContext, stack
from numga.algebras import VGA3D
from examples.animation import save_animation
from examples.mechanics.earthquake import render
from examples.mechanics.earthquake.core import Mechanism, directions

ga = VGA3D
mv = NumpyContext(ga).multivector
Vector = ga.gatype.vector()
Moment = ga.gatype((Vector, Vector))                       # Vector <- Vector

north, east, down = mv.basis()                             # [] Vector each
# The faults: their names, strikes, dips and rakes, in degrees, and the seismic moment of each.
NAMES = ["strike-slip", "normal", "thrust", "oblique"]
STRIKES = np.array([30.0, 30.0, 30.0, 30.0])
DIPS = np.array([90.0, 60.0, 30.0, 50.0])
RAKES = np.array([0.0, -90.0, 90.0, 45.0])
MOMENT = 1.0
# The sweep: one fault's strike and dip, and the frames of its rake's turn.
SWEEP_STRIKE, SWEEP_DIP = 30.0, 50.0
FRAMES = 72
# The grid of directions: rings from straight down, spokes around the vertical.
RINGS = 60
SPOKES = 120

## 1. A fault is a rotor

Space here is north, east and down, as in seismology. A fault is a plane in the ground and a direction of slip within it, given by three angles: the strike, the compass direction of the fault's horizontal line; the dip, how steeply it plunges, to the right of the strike; and the rake, the direction of slip within the plane. Each angle is a turn, and the three compose into one rotor that carries north onto the slip and up onto the fault's normal.

In [ ]:
def fault_frame(strikes, dips, rakes):
    """The turn by the rake in the horizontal plane, then the dip about north, then the strike from north."""
    rake = ((north ^ east) * (rakes / 2)).exp()                               # [...] Rotor: north towards -east
    dip = ((east ^ down) * (-dips / 2)).exp()                                 # [...] Rotor: east towards down
    strike = ((north ^ east) * (-strikes / 2)).exp()                          # [...] Rotor: north towards east
    return strike * dip * rake


frames = fault_frame(np.deg2rad(STRIKES), np.deg2rad(DIPS), np.deg2rad(RAKES))   # [faults] Rotor
normals, slips = frames >> -down, frames >> north                              # [faults] Vector each

## 2. The moment of a slip

The slip pushes the ground on one side of the fault along the slip and the other side back: a couple, the slip times the normal's component of whatever direction it is given. A couple alone would set the ground turning; how much, its curl, is the map wedged with the open vector it is applied to and contracted, and comes out as the plane of shear `normal ^ slip`. The ground does not turn, so a mirrored couple balances it, and the moment is the sum, with no curl. It is symmetric in the normal and the slip: the fault and the plane perpendicular to its slip, the auxiliary plane, have the same moment.

In [ ]:
def double_couple(normals, slips, moments):
    """The slip's couple across the fault, and the mirrored couple that keeps the ground from turning."""
    return moments * (slips * (normals | Vector) + normals * (slips | Vector))   # [...] Moment


def curl(moment):
    """How much a moment would turn the ground."""
    return (Vector ^ moment(Vector)).contract(1, 2)                           # [...] Bivector


single = slips * (normals | Vector)                                           # [faults] Moment: one couple
moment = double_couple(normals, slips, MOMENT)                                # [faults] Moment
auxiliary = double_couple(slips, normals, MOMENT)                             # [faults] Moment: the same

## 3. First motions and the beachball

Along a direction, the P wave's first motion is the component of the moment's response along that direction: positive for a push away from the source, negative for a pull. The S wave carries the rest of the response, across the direction. Over the lower hemisphere, two turns per direction, the pushes are shaded and the nodal lines where the P wave is silent are the fault and the auxiliary plane. The moment's eigenvectors are its pressure axis, in the middle of the pulls, and its tension axis, in the middle of the pushes.

In [ ]:
def radiation(moment, directions):
    """The P wave's first motion along each direction, and the S wave's."""
    response = moment(directions)                                             # [...] Vector
    push = directions | response                                              # [...] Scalar
    return push, response - directions * push


def mechanism(frames, normals, slips, moment, grid):
    """The faults with their principal axes and first motions over a grid of directions."""
    values, axes = moment.eigh()                                              # [faults, 3]: pressure, null, tension
    push, shear = radiation(moment[:, None, None], grid)                      # [faults, rings, spokes]
    return Mechanism(frames, normals, slips, moment, values, axes, grid, push, shear)


hemisphere = directions(np.linspace(0.0, np.pi / 2, RINGS), np.linspace(0.0, 2 * np.pi, SPOKES + 1))   # [RINGS, SPOKES + 1] Vector
faults = mechanism(frames, normals, slips, moment, hemisphere)
render.draw_mechanisms(faults, NAMES);

## 4. From the moment back to the fault

The moment sends the tension axis to itself and the pressure axis to its opposite, and the normal and slip are their sum and difference, over the square root of two. Which is which, the moment cannot say: both choices rebuild it, and only other evidence, such as the trace of the fault at the surface or the spread of aftershocks, picks the fault from the auxiliary plane.

In [ ]:
def fault_planes(moment):
    """The two faults with this moment, each fault's normal the other's slip."""
    _, axes = moment.eigh()                                                   # [..., 3] Vector
    pressure, tension = axes[..., 0], axes[..., 2]
    first, second = (tension + pressure) / np.sqrt(2), (tension - pressure) / np.sqrt(2)
    return stack([first, second], axis=-1), stack([second, first], axis=-1)   # [..., 2] Vector each


planes, directions_of_slip = fault_planes(moment)                             # [faults, 2] Vector each
rebuilt = double_couple(planes, directions_of_slip, MOMENT)                   # [faults, 2] Moment

## 5. Turning the rake

Keeping the fault and turning its slip once around, the ball goes from a normal fault, through strike-slip, to a thrust and back. In space, each direction is drawn as far as its P wave's first motion: four lobes, two pushing and two pulling, with the fault in between.

In [ ]:
rakes = np.linspace(-np.pi, np.pi, FRAMES, endpoint=False)
sweep = fault_frame(np.deg2rad(SWEEP_STRIKE), np.deg2rad(SWEEP_DIP), rakes)    # [FRAMES] Rotor
sweep_normals, sweep_slips = sweep >> -down, sweep >> north                    # [FRAMES] Vector each
sweep_moment = double_couple(sweep_normals, sweep_slips, MOMENT)               # [FRAMES] Moment
sphere = directions(np.linspace(0.0, np.pi, RINGS), np.linspace(0.0, 2 * np.pi, SPOKES + 1))   # [RINGS, SPOKES + 1] Vector
balls = mechanism(sweep, sweep_normals, sweep_slips, sweep_moment, hemisphere)
lobes = mechanism(sweep, sweep_normals, sweep_slips, sweep_moment, sphere)
display(Image(filename=save_animation(render.animate_sweep(balls, lobes), "earthquake_rake", 80)))

In [ ]:
# checks
# A slip neither swells nor shrinks the ground and has no third axis.
np.testing.assert_allclose(moment.trace().to_array(), 0.0, atol=1e-14)
np.testing.assert_allclose(moment.det().to_array(), 0.0, atol=1e-14)
np.testing.assert_allclose(faults.values.to_array(), np.array([-MOMENT, 0.0, MOMENT]) * np.ones((len(NAMES), 1)), atol=1e-13)
# One couple turns the ground by the plane of shear; the double couple does not.
np.testing.assert_allclose(curl(single).kernel, (normals ^ slips).kernel, atol=1e-14)
np.testing.assert_allclose(curl(moment).kernel, 0.0, atol=1e-14)
# The fault and the auxiliary plane give the same moment, and both planes found from it rebuild it.
np.testing.assert_allclose(auxiliary.kernel, moment.kernel, atol=1e-14)
np.testing.assert_allclose(rebuilt.kernel, moment[:, None].broadcast_to(rebuilt.shape).kernel, atol=1e-13)
np.testing.assert_allclose((planes | normals[:, None]).abs().to_array().max(axis=-1), 1.0, rtol=1e-12)
# Half a turn of rake reverses the slip and flips the moment.
np.testing.assert_allclose((sweep_moment[FRAMES // 2:] + sweep_moment[:FRAMES // 2]).kernel, 0.0, atol=1e-14)